# Module 6 — Multi-Omics Integration (CNV + RNA)

**Biological question:** Does copy-number variation (CNV) at the DNA level influence gene expression at the RNA level? And can combining CNV and RNA data into a single embedding reveal subtype structure better than RNA alone?

**Background:** In breast cancer, ERBB2 (HER2) is frequently amplified at the DNA level in HER2+ tumours, leading to overexpression of the HER2 protein. This makes it a strong positive control: we expect high CNV and high RNA expression to co-occur in HER2+ samples.

**Methods:**
- **CNV data source:** GISTIC2 thresholded copy-number calls from GDC hub. Values: -2 (deep deletion), -1 (shallow deletion), 0 (neutral), 1 (gain), 2 (amplification).
- **CNV–RNA correlation:** Per-gene Pearson correlation between GISTIC2 call and log-normalised RNA expression across all shared samples.
- **Joint UMAP:** Each omic scaled to zero mean/unit variance, then concatenated and projected to 2D.

> **Note:** This notebook requires the CNV file to have been downloaded with `make fetch`.

In [ ]:
from pathlib import Path
assert Path("../data/raw/TCGA-BRCA.gistic2thresholded.tsv.gz").exists(), (
    "CNV file not found. Run `make fetch` first to download TCGA-BRCA.gistic2thresholded.tsv.gz"
)

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.constants import InterimFile
from src.paths import interim_file
from src.features.multiomics import load_align_cnv, cnv_rna_correlation, joint_embedding
from src.visualization.embeddings import plot_embedding

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load Data

In [ ]:
X_log = pd.read_csv(interim_file(InterimFile.X), index_col=0)
labels = pd.read_csv(interim_file(InterimFile.LABELS), index_col=0).squeeze()

cnv = load_align_cnv(X_index=X_log.index)
print(f"CNV:  {cnv.shape[0]} samples × {cnv.shape[1]} genes")
print(f"RNA:  {X_log.shape[0]} samples × {X_log.shape[1]} genes")

## Distribution of CNV Values per Subtype

GISTIC2 values: -2=deep del, -1=shallow del, 0=neutral, 1=gain, 2=amplification. HER2+ tumours should show enrichment of +2 (amplification) at the ERBB2 locus.

In [ ]:
shared_samples = cnv.index.intersection(labels.index)
cnv_flat = cnv.loc[shared_samples].values.ravel()

cnv_with_labels = pd.DataFrame({"CNV": cnv_flat})
unique_vals, counts = np.unique(cnv_flat[~np.isnan(cnv_flat)], return_counts=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(unique_vals, counts / counts.sum() * 100, color="#4878cf")
ax.set_xticks([-2, -1, 0, 1, 2])
ax.set_xticklabels(["Deep del", "Shallow del", "Neutral", "Gain", "Amplification"])
ax.set_ylabel("% of all gene-sample pairs")
ax.set_title("GISTIC2 CNV value distribution (all genes, all samples)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## ERBB2: CNV vs Log-RNA Expression (HER2 Control)

ERBB2 (Ensembl `ENSG00000141736`) encodes the HER2 receptor. In HER2+ tumours it is amplified at the DNA level, driving overexpression. We expect a positive correlation between CNV value and RNA expression for this gene, with HER2+ samples clustering in the upper-right quadrant.

In [ ]:
ERBB2_BASE = "ENSG00000141736"

erbb2_cnv_cols = [c for c in cnv.columns if c.startswith(ERBB2_BASE)]
erbb2_rna_cols = [c for c in X_log.columns if c.startswith(ERBB2_BASE)]

if erbb2_cnv_cols and erbb2_rna_cols:
    shared = cnv.index.intersection(X_log.index).intersection(labels.index)
    erbb2_cnv = cnv.loc[shared, erbb2_cnv_cols[0]]
    erbb2_rna = X_log.loc[shared, erbb2_rna_cols[0]]
    subtype_labels = labels.loc[shared]

    categories = sorted(subtype_labels.unique())
    cmap = plt.get_cmap("tab10", len(categories))

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, cat in enumerate(categories):
        mask = subtype_labels == cat
        ax.scatter(erbb2_cnv[mask], erbb2_rna[mask], label=cat, color=cmap(i),
                   s=15, alpha=0.7, linewidths=0)
    ax.set_xlabel("ERBB2 CNV (GISTIC2 thresholded)")
    ax.set_ylabel("ERBB2 log₂(counts+1)")
    ax.set_title("ERBB2: CNV vs RNA expression", fontsize=11, fontweight="bold")
    ax.legend(markerscale=2, fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print(f"ERBB2 not found in one or both omics. CNV cols: {erbb2_cnv_cols}, RNA cols: {erbb2_rna_cols}")

## CNV–RNA Correlation Histogram

Most genes show weak CNV–RNA correlation (values near 0), reflecting the many regulatory layers between DNA copy number and mRNA abundance. Genes with strong positive correlation (r > 0.4) are those where copy-number amplification directly drives transcription — ERBB2 is a canonical example.

In [ ]:
corr = cnv_rna_correlation(cnv=cnv, rna_log=X_log)
corr_clean = corr.dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(corr_clean, bins=60, color="#4878cf", edgecolor="white", linewidth=0.3)
ax.axvline(corr_clean.median(), color="#d62728", lw=1.5, ls="--", label=f"median = {corr_clean.median():.3f}")
ax.set_xlabel("Pearson r (CNV vs log-RNA)")
ax.set_ylabel("Number of genes")
ax.set_title("Per-gene CNV–RNA correlation", fontsize=11, fontweight="bold")
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

print(f"Top 10 positively correlated genes:")
print(corr_clean.nlargest(10).to_string())

## Joint CNV + RNA UMAP

Combining both omics and projecting to 2D: do the PAM50 subtypes still separate? If yes, the joint representation captures complementary biological signal from both layers.

In [ ]:
joint_emb = joint_embedding(cnv=cnv, rna_log=X_log)
shared_labels = labels.loc[joint_emb.index]

fig, ax = plt.subplots(figsize=(8, 6))
plot_embedding(
    emb=joint_emb,
    labels=shared_labels,
    title="Joint CNV + RNA UMAP — coloured by PAM50 subtype",
    ax=ax,
)
plt.tight_layout()
plt.show()